In [ ]:
!pip install minatar

# 02_offline_rl_clean.ipynb
Offline RL (BCQ) for MiniAtari Breakout
Contains: load expert dataset, BCQ training, evaluation, GIF generation.


In [ ]:
# Установка
!pip install --quiet minatar imageio-ffmpeg

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/rl-final-project"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "dqn"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "imitation"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "offline_rl"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "visuals", "random_breakout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "visuals", "expert_rollout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "visuals", "bc_rollout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "visuals", "cql_bcq_rollout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "plots"), exist_ok=True)

print("Project dir:", DRIVE_PROJECT_DIR)

Mounted at /content/drive
Project dir: /content/drive/MyDrive/rl-final-project


In [ ]:
import random, time, pickle
from collections import deque, namedtuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import imageio
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
import minatar
env = minatar.Environment("breakout")
print("env.state_shape():", env.state_shape())
print("env.num_actions():", env.num_actions())

env.state_shape(): [10, 10, 4]
env.num_actions(): 6


In [ ]:
def preprocess_state(st):
    arr = np.array(st, dtype=np.float32)
    # defensive fixes for odd shapes
    if arr.ndim == 1:
        if arr.size == 1:
            arr = np.full((10,10), arr[0], dtype=np.float32)
        else:
            try:
                arr = arr.reshape(10,10)
            except:
                arr = np.zeros((10,10), dtype=np.float32)
    if arr.ndim == 3:
        arr = np.sum(arr, axis=2)
    mn, mx = arr.min(), arr.max()
    rng = mx - mn if mx > mn else 1.0
    arr = (arr - mn) / rng
    flat = arr.flatten()
    if flat.shape[0] != 100:
        flat = np.resize(flat, 100)
    return flat

def state_to_numpy(st):
    return np.array(st, dtype=np.float32)

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
    def push(self, *args):
        self.buffer.append(Transition(*args))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))
    def __len__(self):
        return len(self.buffer)

In [ ]:
class QNetworkMLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, n_actions):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, n_actions))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

In [ ]:
class BehaviorClone(nn.Module):
    def __init__(self, state_dim=100, hidden_sizes=[256,128], num_actions=6):
        super().__init__()
        layers = []
        last = state_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, num_actions))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

In [ ]:
dataset_path = os.path.join(DRIVE_PROJECT_DIR, "dqn", "expert_dataset.pkl")
with open(dataset_path, "rb") as f:
    expert_data = pickle.load(f)
print("Loaded transitions:", len(expert_data))

states, actions, rewards, next_states, dones = [], [], [], [], []
for s,a,r,ns,d in expert_data:
    states.append(s)
    actions.append(a)
    rewards.append(r)
    next_states.append(ns)
    dones.append(d)

states = np.array(states, dtype=np.float32)
actions = np.array(actions, dtype=np.int64)
rewards = np.array(rewards, dtype=np.float32)
next_states = np.array(next_states, dtype=np.float32)
dones = np.array(dones, dtype=np.float32)

print("states shape:", states.shape)
print("actions shape:", actions.shape)

Loaded transitions: 50000
states shape: (50000, 100)
actions shape: (50000,)


In [ ]:
state_dim = 100
num_actions = env.num_actions()  # use env's action count

# Behavior cloning (π_β)
bc_model = BehaviorClone(state_dim=state_dim, hidden_sizes=[256,128], num_actions=num_actions).to(device)

# Q networks (same architecture as DQN: [256,128])
q_model = QNetworkMLP(state_dim, [256,128], num_actions).to(device)
target_q = QNetworkMLP(state_dim, [256,128], num_actions).to(device)
target_q.load_state_dict(q_model.state_dict())

print("Models ready: bc_model, q_model, target_q")

Models ready: bc_model, q_model, target_q


In [ ]:
# Hyperparameters
bc_lr = 1e-3
q_lr = 1e-3
batch_size = 512
discount = 0.99
num_updates = 2000   # number of gradient steps (adjustable)
tau = 0.005          # soft update
threshold = 0.25     # action probability threshold for BCQ mask

optim_bc = optim.Adam(bc_model.parameters(), lr=bc_lr)
optim_q = optim.Adam(q_model.parameters(), lr=q_lr)

dataset_size = states.shape[0]
print("Dataset size:", dataset_size)

loss_log = []
bc_loss_log = []

for it in range(1, num_updates+1):
    idx = np.random.randint(0, dataset_size, batch_size)
    s = torch.tensor(states[idx], dtype=torch.float32).to(device)
    a = torch.tensor(actions[idx], dtype=torch.long).to(device)
    r = torch.tensor(rewards[idx], dtype=torch.float32).unsqueeze(1).to(device)
    ns = torch.tensor(next_states[idx], dtype=torch.float32).to(device)
    d = torch.tensor(dones[idx], dtype=torch.float32).unsqueeze(1).to(device)

    # Train BC model (behavioral policy)
    logits = bc_model(s)
    bc_loss = nn.CrossEntropyLoss()(logits, a)
    optim_bc.zero_grad()
    bc_loss.backward()
    optim_bc.step()

    # Train Q
    with torch.no_grad():
        next_logits = bc_model(ns)
        probs = torch.softmax(next_logits, dim=1)
        mask = (probs > threshold).float()

        q_next = target_q(ns)  # (B, A)
        q_next_masked = q_next * mask + (-1e9) * (1 - mask)
        q_max = q_next_masked.max(1)[0].unsqueeze(1)

        target = r + discount * (1 - d) * q_max

    q_values = q_model(s)
    q_taken = q_values.gather(1, a.unsqueeze(1))

    q_loss = nn.MSELoss()(q_taken, target)

    optim_q.zero_grad()
    q_loss.backward()
    optim_q.step()

    # Soft update
    for tp, qp in zip(target_q.parameters(), q_model.parameters()):
        tp.data.copy_(tp.data * (1 - tau) + qp.data * tau)

    loss_log.append(q_loss.item())
    bc_loss_log.append(bc_loss.item())

    if it % 200 == 0:
        print(f"Iter {it}/{num_updates} | bc_loss={bc_loss.item():.4f} | q_loss={q_loss.item():.4f}")

Dataset size: 50000
Iter 200/2000 | bc_loss=nan | q_loss=nan
Iter 400/2000 | bc_loss=nan | q_loss=nan
Iter 600/2000 | bc_loss=nan | q_loss=nan
Iter 800/2000 | bc_loss=nan | q_loss=nan
Iter 1000/2000 | bc_loss=nan | q_loss=nan
Iter 1200/2000 | bc_loss=nan | q_loss=nan
Iter 1400/2000 | bc_loss=nan | q_loss=nan
Iter 1600/2000 | bc_loss=nan | q_loss=nan
Iter 1800/2000 | bc_loss=nan | q_loss=nan
Iter 2000/2000 | bc_loss=nan | q_loss=nan


In [ ]:
bcq_model_path = os.path.join(DRIVE_PROJECT_DIR, "offline_rl", "bcq_q_model.pth")
bc_model_path = os.path.join(DRIVE_PROJECT_DIR, "offline_rl", "bcq_behavior_model.pth")
torch.save(q_model.state_dict(), bcq_model_path)
torch.save(bc_model.state_dict(), bc_model_path)
print("Saved BCQ Q model:", bcq_model_path)
print("Saved BCQ behavior model:", bc_model_path)

# Save loss logs
np.save(os.path.join(DRIVE_PROJECT_DIR, "offline_rl", "bcq_q_loss.npy"), np.array(loss_log))
np.save(os.path.join(DRIVE_PROJECT_DIR, "offline_rl", "bcq_bc_loss.npy"), np.array(bc_loss_log))

Saved BCQ Q model: /content/drive/MyDrive/rl-final-project/offline_rl/bcq_q_model.pth
Saved BCQ behavior model: /content/drive/MyDrive/rl-final-project/offline_rl/bcq_behavior_model.pth


In [ ]:
def bcq_policy(state, threshold_local=threshold):
    s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = bc_model(s)
        probs = torch.softmax(logits, dim=1)
        mask = (probs > threshold_local).float()
        q = q_model(s)
        q_masked = q * mask + (-1e9) * (1 - mask)
        action = int(torch.argmax(q_masked, dim=1).item())
    return action

In [ ]:
def generate_bcq_gif(env, steps=400, scale=20, save_subdir="visuals/cql_bcq_rollout"):
    import minatar
    out_dir = os.path.join(DRIVE_PROJECT_DIR, save_subdir)
    os.makedirs(out_dir, exist_ok=True)
    env.reset()
    s = preprocess_state(env.state())
    frames = []
    for t in range(steps):
        a = bcq_policy(s)
        reward, done = env.act(a)
        raw = state_to_numpy(env.state())
        img = raw.sum(axis=2) if raw.ndim == 3 else raw
        mn, mx = img.min(), img.max()
        rng = mx - mn if mx > mn else 1.0
        img_norm = ((img - mn) / rng * 255).astype(np.uint8)
        frame = Image.fromarray(img_norm).resize((img_norm.shape[0]*scale, img_norm.shape[1]*scale), Image.NEAREST)
        frames.append(np.array(frame))
        s = preprocess_state(env.state())
    gif_path = os.path.join(out_dir, "bcq_rollout.gif")
    imageio.mimsave(gif_path, frames, fps=12)
    print("Saved BCQ GIF:", gif_path)
    return gif_path

# Run it (uncomment to generate)
# gif_path = generate_bcq_gif(env)
# gif_path

In [ ]:
def eval_policy(env, policy_fn, episodes=10, steps_per_ep=400):
    scores = []
    for ep in range(episodes):
        env.reset()
        s = preprocess_state(env.state())
        total = 0
        for t in range(steps_per_ep):
            a = policy_fn(s)
            r, done = env.act(a)
            total += r
            s = preprocess_state(env.state())
        scores.append(total)
    return scores

# random policy
def random_policy(s): return random.randint(0, num_actions-1)

# load expert (DQN) — ensure same architecture
expert_net = QNetworkMLP(state_dim, [256,128], num_actions).to(device)
expert_path = os.path.join(DRIVE_PROJECT_DIR, "dqn", "best_model.pth")
expert_net.load_state_dict(torch.load(expert_path, map_location=device))
expert_net.eval()
def expert_policy(s):
    with torch.no_grad():
        x = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
        return int(torch.argmax(expert_net(x)).item())

# evaluate
import minatar
env_eval = minatar.Environment("breakout")

scores_random = eval_policy(env_eval, random_policy, episodes=10)
scores_expert = eval_policy(env_eval, expert_policy, episodes=10)
scores_bcq = eval_policy(env_eval, bcq_policy, episodes=10)

print("Random: mean {:.2f} ± {:.2f}".format(np.mean(scores_random), np.std(scores_random)))
print("BCQ:    mean {:.2f} ± {:.2f}".format(np.mean(scores_bcq), np.std(scores_bcq)))
print("Expert: mean {:.2f} ± {:.2f}".format(np.mean(scores_expert), np.std(scores_expert)))

Random: mean 0.40 ± 0.66
BCQ:    mean 0.60 ± 0.49
Expert: mean 0.40 ± 0.49


In [ ]:
# boxplot and bar plot for results (quick)
plt.figure(figsize=(6,4))
plt.boxplot([scores_random, scores_bcq, scores_expert], labels=["Random","BCQ","Expert"])
plt.title("Policy comparison")
plt.savefig(os.path.join(DRIVE_PROJECT_DIR, "plots", "offline_policy_comparison.png"), dpi=150)
plt.close()
print("Saved comparison plot.")

Saved comparison plot.


/tmp/ipython-input-3096599869.py:3: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([scores_random, scores_bcq, scores_expert], labels=["Random","BCQ","Expert"])


In [ ]:
comparison_dir = os.path.join(DRIVE_PROJECT_DIR, "plots", "comparison")
os.makedirs(comparison_dir, exist_ok=True)
print("Saving comparison plots to:", comparison_dir)

Saving comparison plots to: /content/drive/MyDrive/rl-final-project/plots/comparison


In [ ]:
import minatar

env_eval = minatar.Environment("breakout")

# === Evaluate all policies ===
scores_random = eval_policy(env_eval, random_policy, episodes=20)
scores_expert = eval_policy(env_eval, expert_policy, episodes=20)
scores_bcq = eval_policy(env_eval, bcq_policy, episodes=20)


print("Random:", np.mean(scores_random), np.std(scores_random))
print("Expert:", np.mean(scores_expert), np.std(scores_expert))
print("BCQ:",    np.mean(scores_bcq), np.std(scores_bcq))

Random: 0.5 0.5
Expert: 0.65 0.47696960070847283
BCQ: 0.4 0.48989794855663565


In [ ]:
plt.figure(figsize=(7,5))
plt.boxplot(
    [scores_random, scores_bcq, scores_expert],
    labels=["Random", "BCQ", "DQN Expert"]
)
plt.title("Policy Performance Comparison (20 episodes)")
plt.ylabel("Total Episode Reward")
plt.grid(True)

path_box = os.path.join(comparison_dir, "boxplot_policies.png")
plt.savefig(path_box, dpi=150)
plt.close()
print("Saved boxplot:", path_box)

Saved boxplot: /content/drive/MyDrive/rl-final-project/plots/comparison/boxplot_policies.png


/tmp/ipython-input-1524719831.py:2: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(


In [ ]:
means = [
    np.mean(scores_random),
    np.mean(scores_bcq),
    np.mean(scores_expert)
]
stds = [
    np.std(scores_random),
    np.std(scores_bcq),
    np.std(scores_expert)
]

labels = ["Random", "BCQ", "DQN Expert"]

plt.figure(figsize=(7,5))
plt.bar(labels, means, yerr=stds, capsize=8)
plt.title("Average Performance with Standard Deviation")
plt.ylabel("Average Reward")

path_bar = os.path.join(comparison_dir, "bar_mean_std.png")
plt.savefig(path_bar, dpi=150)
plt.close()

print("Saved bar chart:", path_bar)

Saved bar chart: /content/drive/MyDrive/rl-final-project/plots/comparison/bar_mean_std.png


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Policy": ["Random", "BCQ", "DQN Expert"],
    "Mean Reward": means,
    "Std Reward": stds,
})
csv_path = os.path.join(comparison_dir, "policy_comparison.csv")
df.to_csv(csv_path, index=False)

df

,Policy,Mean Reward,Std Reward
0,Random,0.50,0.500000
1,BCQ,0.40,0.489898
2,DQN Expert,0.65,0.476970
